In [1]:
import pandas as pd
from scipy.stats import friedmanchisquare
from itertools import combinations
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

# Load data
df = pd.read_csv("Output_extraction/ai_grading_final_v3.csv")

# Clean columns
df["answer_key_id"] = df["answer_key_id"].astype(str).str.strip()
df["prompt_type"] = df["prompt_type"].astype(str).str.strip()

# Create outcome variable
df["absolute_error"] = abs(
    df["ai_estimated_mistakes"] - df["true_mistakes"]
)

# Prompt order
prompt_order = [
    "very_pessimistic",
    "pessimistic",
    "neutral",
    "confident",
    "very_confident"
]

# If answer_key_id is not unique, create a repetition index
df["rep"] = df.groupby(
    ["answer_key_id", "true_mistakes", "prompt_type"]
).cumcount()

# Convert to wide format
df_wide = df.pivot_table(
    index=["answer_key_id", "true_mistakes", "rep"],
    columns="prompt_type",
    values="absolute_error",
    aggfunc="first"
)

# Keep only complete rows with all five prompt types
df_wide = df_wide[prompt_order].dropna()

# Friedman test
friedman_result = friedmanchisquare(
    df_wide["very_pessimistic"],
    df_wide["pessimistic"],
    df_wide["neutral"],
    df_wide["confident"],
    df_wide["very_confident"]
)

print("Friedman test")
print("Statistic:", friedman_result.statistic)
print("p-value:", friedman_result.pvalue)

Friedman test
Statistic: 10.559709733388633
p-value: 0.03198436795704993


In [ ]:
posthoc_results = []

for p1, p2 in combinations(prompt_order, 2):
    test = wilcoxon(
        df_wide[p1],
        df_wide[p2],
        zero_method="wilcox",
        alternative="two-sided"
    )

    posthoc_results.append({
        "comparison": f"{p1} vs {p2}",
        "p_value": test.pvalue
    })

posthoc_df = pd.DataFrame(posthoc_results)

# Holm correction
posthoc_df["p_holm"] = multipletests(
    posthoc_df["p_value"],
    method="holm"
)[1]

print(posthoc_df)

In [2]:
import pandas as pd
from scipy.stats import friedmanchisquare
from itertools import combinations
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

# Load data
df = pd.read_csv("Output_extraction/ai_grading_final_v3.csv")

# Clean columns
df["answer_key_id"] = df["answer_key_id"].astype(str).str.strip()
df["prompt_type"] = df["prompt_type"].astype(str).str.strip()

# Create signed error variable
df["signed_error"] = (
    df["ai_estimated_mistakes"] - df["true_mistakes"]
)

# Prompt order
prompt_order = [
    "very_pessimistic",
    "pessimistic",
    "neutral",
    "confident",
    "very_confident"
]

# If answer_key_id is not unique, create a repetition index
df["rep"] = df.groupby(
    ["answer_key_id", "true_mistakes", "prompt_type"]
).cumcount()

# Convert to wide format
df_wide = df.pivot_table(
    index=["answer_key_id", "true_mistakes", "rep"],
    columns="prompt_type",
    values="signed_error",
    aggfunc="first"
)

# Keep only complete rows with all five prompt types
df_wide = df_wide[prompt_order].dropna()

# Friedman test
friedman_result = friedmanchisquare(
    df_wide["very_pessimistic"],
    df_wide["pessimistic"],
    df_wide["neutral"],
    df_wide["confident"],
    df_wide["very_confident"]
)

print("Friedman test for signed error")
print("Statistic:", friedman_result.statistic)
print("p-value:", friedman_result.pvalue)

Friedman test for signed error
Statistic: 6.900315117973171
p-value: 0.14125082660224858
